# Exploration de la source BCEAO

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)  
**Module :** 1 - Collecte  
**Source étudiée :** BCEAO (Banque Centrale des États de l'Afrique de l'Ouest)

## Objectif de ce carnet

Avant d'écrire le collecteur définitif, je veux comprendre comment cette source expose ses
appels d'offres : peut-on y accéder simplement, quels champs sont disponibles, et sous quelle
forme. Je documente toute ma démarche, y compris les vérifications et les choix, pour garder
une trace de pourquoi le code final est écrit ainsi.

## 1. Comment j'en suis arrivée à la BCEAO

Je suis partie de la liste de sources fournie par l'entreprise. Avant de coder, j'ai testé les
URLs pour voir lesquelles étaient vraiment exploitables :

- **Banques et assurances privées** (Attijari, UIB, ATB, Maghrebia...) : pas de page publique
  d'appels d'offres. Elles passent par une plateforme fournisseurs sur inscription, ou ne
  publient rien en ligne. Non collectables sans compte.
- **TUNEPS** (marchés publics tunisiens) : accès via certificat TUNTRUST, trop complexe pour
  un premier essai.
- **ANPR** : beaucoup de bruit (mélange appels d'offres, bourses, recrutements).
- **Plusieurs URLs renvoyaient une erreur 404** (liens périmés).

La **BCEAO** répondait correctement et affichait une vraie liste d'appels d'offres publique et
structurée. Je la prends donc comme **source pilote** : si toute la chaîne fonctionne dessus,
je pourrai l'étendre aux autres sources ensuite.

Remarque : la BCEAO est la banque centrale de l'Afrique de l'Ouest (UEMOA). Pertinente
seulement si le périmètre du projet couvre cette région, ce que la liste fournie semble
confirmer. À valider avec l'encadrante.

## 2. Récupération de la page

Je récupère le HTML de la page des appels d'offres. J'ajoute un `User-Agent` de navigateur,
parce que beaucoup de sites bloquent les requêtes qui n'en ont pas (elles ressemblent à des
robots).

In [1]:
import requests
import re
import json
from bs4 import BeautifulSoup

URL = "https://www.bceao.int/fr/appels-offres/appels-offres-marches-publics-achats"
BASE = "https://www.bceao.int"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124.0 Safari/537.36"}

reponse = requests.get(URL, headers=HEADERS, timeout=30)
print("Code HTTP :", reponse.status_code)
print("Taille de la page :", len(reponse.content), "octets")

Code HTTP : 200
Taille de la page : 132597 octets


## 3. Le contenu est-il déjà dans le HTML brut ?

Première question avant tout : puis-je me contenter de `requests`, ou le contenu est-il chargé
en JavaScript (ce qui imposerait un navigateur automatisé comme Playwright, plus lourd) ?

Le test : est-ce que les liens d'annonces sont présents dans le HTML que le serveur m'a envoyé ?
- Si oui → rendu côté serveur → `requests` suffit.
- Si non → chargé en JavaScript → il faudrait Playwright.

(C'est utile car sur un autre portail, l'ONMP, le tableau était vide dans le HTML brut : tout
était chargé en JavaScript.)

In [2]:
# re.findall cherche, dans le HTML BRUT, tous les bouts d'URL ayant la forme d'un lien d'annonce.
# 'reponse.text' est exactement ce que le serveur a renvoyé (aucun JavaScript exécuté).
liens_bruts = re.findall(r"/fr/appels-offres/[a-z0-9\-]{15,}", reponse.text)
nb = len(set(liens_bruts))
print(f"Liens d'annonces trouvés dans le HTML brut : {nb}\n")

if nb > 0:
    print("=> CAS 1 : le contenu est déjà dans le HTML. requests suffit.")
else:
    print("=> CAS 2 : rien dans le HTML brut. Contenu en JavaScript -> il faudrait Playwright.")

Liens d'annonces trouvés dans le HTML brut : 35

=> CAS 1 : le contenu est déjà dans le HTML. requests suffit.


## 4. La page est organisée en sections

En regardant la page, je vois trois sections, chacune introduite par un titre `<h2>` :
« Appel d'offres En cours », « Appel d'offres Clos », « Résultats ». Les annonces d'une section
sont dans le `<div>` qui suit immédiatement son `<h2>` (son frère).

Je vérifie combien d'annonces chaque section contient dans le HTML brut.

In [3]:
soup = BeautifulSoup(reponse.text, "html.parser")

for h2 in soup.find_all("h2"):
    titre = " ".join(h2.get_text().split())
    if any(mot in titre.lower() for mot in ("en cours", "clos", "résultat", "resultat")):
        frere = h2.find_next_sibling()
        if frere is None:
            continue
        liens = frere.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
        avec_date = [a for a in liens if "ublié le" in a.get_text()]
        print(f"{titre:30} -> {len(avec_date)} annonce(s) dans le HTML brut")

Appel d’offres En cours        -> 22 annonce(s) dans le HTML brut
Appel d’offres Clos            -> 1 annonce(s) dans le HTML brut
Résultats                      -> 0 annonce(s) dans le HTML brut


**Observation importante :** la section « En cours » contient bien ses annonces dans le HTML
brut, mais « Clos » n'en montre presque aucune. En réalité, sur le site, la section « Clos » est
remplie en JavaScript au moment où on l'ouvre dans un navigateur ; `requests` ne la voit donc pas.

Ce n'est pas un problème : pour la veille, ce qui m'intéresse, ce sont les annonces **ouvertes**
(« En cours »), parfaitement accessibles ici. Je vais donc cibler uniquement cette section. Les
annonces closes (en JavaScript) demanderaient Playwright et ont peu d'intérêt pour la prospection.

Autre point : le `<h2>` est mon **point d'ancrage fiable**. Je ne peux pas me baser sur la classe
CSS de la section, car elle est générée aléatoirement (elle change à chaque rechargement).

## 5. Inspection d'une annonce : où sont les données ?

Avant d'écrire l'extraction, je regarde le HTML d'une seule annonce pour comprendre comment elle
est structurée (quelles balises, où sont le titre, les dates, le lien).

In [4]:
# Un exemple d'annonce (lien ayant la forme attendue ET contenant "Publié le")
exemple = next(a for a in soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
               if "ublié le" in a.get_text())

print("Balise :", exemple.name)
print("Lien (href) :", exemple.get("href"))
print("\nTexte visible de l'annonce :")
print(repr(" ".join(exemple.get_text().split())))

Balise : a
Lien (href) : https://www.bceao.int/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de

Texte visible de l'annonce :
'Publié le 16 juin 2026 Date limite le 30 Juillet 2026 Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin'


Je constate que **tout est dans le texte d'un seul lien `<a>`** : la date de publication, la
date limite et le titre sont collés ensemble, pas répartis dans des balises distinctes. Les
champs suivent toujours le même modèle :

```
Publié le <date publication> <référence éventuelle> Date limite le <date limite> <titre>
```

Comme les champs ne sont pas séparés par des balises, je devrai utiliser une **expression
régulière** sur ce texte pour les découper. Le lien `href`, lui, me donnera l'URL de la fiche.

## 6. Pourquoi je filtre sur « Publié le »

Tous les liens de forme `/fr/appels-offres/...` ne sont pas de vraies annonces : il y a aussi le
lien vers la page elle-même, des appels à candidatures (formation), ou des opérations de marché
monétaire. Ces liens ont la bonne **forme** d'URL, et certains contiennent même les mots « appel
d'offres », mais ils n'ont pas de date « Publié le ».

Filtrer sur la présence de « Publié le » est donc plus fiable que filtrer sur les mots-clés : je
cible la **structure** d'une vraie annonce, pas son vocabulaire. Je liste ci-dessous les liens
écartés pour le vérifier.

In [5]:
liens = soup.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
ecartes = [a for a in liens if "ublié le" not in a.get_text()]

vus = set()
print("Liens écartés (forme d'annonce mais pas de 'Publié le') :\n")
for a in ecartes:
    href = a.get("href")
    if href not in vus:
        vus.add(href)
        print(href, "->", repr(" ".join(a.get_text().split())[:45]))

Liens écartés (forme d'annonce mais pas de 'Publié le') :

/fr/appels-offres/appels-offres-marches-publics-achats -> 'FR'
/fr/appels-offres/appel-candidatures-pour-la-49e-promotion-du-cycle-diplomant-du-cofeb -> 'COFEB'
/fr/appels-offres/marche-mon%C3%A9taire -> 'Appel d’offres du marché'
/fr/appels-offres/appels-offres-marches-titres-publics-prives -> 'Avis d’appel d’offres'


## 7. Récupération des annonces de la section « En cours »

Je définis une petite fonction qui part du `<h2>` « En cours », prend son `<div>` frère, et y
récupère uniquement les vraies annonces (celles avec « Publié le »).

In [6]:
def annonces_de_section(soup, mot_cle):
    """Renvoie les liens d'annonces de la section dont le <h2> contient mot_cle."""
    for h2 in soup.find_all("h2"):
        if mot_cle in " ".join(h2.get_text().split()).lower():
            section = h2.find_next_sibling()
            if section is None:
                return []
            liens = section.find_all("a", href=re.compile(r"/fr/appels-offres/[a-z]"))
            return [a for a in liens if "ublié le" in a.get_text()]
    return []

annonces = annonces_de_section(soup, "en cours")
print("Annonces 'En cours' récupérées :", len(annonces))

Annonces 'En cours' récupérées : 22


## 8. Extraction des champs

Pour chaque annonce, je récupère : date de publication, référence (si présente), date limite,
titre, et le lien vers la fiche de détail.

- **`cle_unique`** : j'utilise le *slug* de l'URL (la fin du lien). Il est toujours présent et
  stable, contrairement à la référence qui manque parfois. Il servira à éviter les doublons.
- **`lien`** : c'est l'URL de la page de détail. **Stratégie simple :** je stocke seulement ce
  lien pour l'instant. La page de détail contient la description complète et les documents
  (cahier des charges en PDF), mais je n'irai les extraire que plus tard, et seulement pour les
  annonces jugées pertinentes (inutile de visiter le détail de 30 marchés hors périmètre).
- **`statut_source`** : « en cours », puisque ces annonces viennent de cette section.

In [7]:
motif = re.compile(
    r"Publié le\s+(?P<pub>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<ref>[A-Z0-9/\-\u00b0N ]+?)?\s*"
    r"Date limite le\s+(?P<lim>\d{1,2}\s+\w+\s+\d{4})\s+"
    r"(?P<titre>.+)",
    re.IGNORECASE,
)

resultats = []
for a in annonces:
    texte = " ".join(a.get_text().split())
    m = motif.search(texte)
    if not m:
        continue  # format inattendu : on passe

    href = a.get("href")
    lien = href if href.startswith("http") else BASE + href

    reference = (m.group("ref") or "").strip()
    if reference and (len(reference) < 4 or reference.lower() == "n"):
        reference = ""

    resultats.append({
        "cle_unique": lien.rstrip("/").split("/")[-1],   # slug de l'URL
        "reference": reference,
        "intitule": m.group("titre").strip(),
        "date_publication": m.group("pub"),
        "delai_soumission": m.group("lim"),
        "lien": lien,                  # page de détail (documents accessibles via ce lien)
        "source": "BCEAO",
        "statut_source": "en cours",
    })

print("Annonces extraites :", len(resultats), "\n")
print(json.dumps(resultats[0], ensure_ascii=False, indent=2))

Annonces extraites : 22 

{
  "cle_unique": "etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de",
  "reference": "",
  "intitule": "Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin",
  "date_publication": "16 juin 2026",
  "delai_soumission": "30 Juillet 2026",
  "lien": "https://www.bceao.int/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de",
  "source": "BCEAO",
  "statut_source": "en cours"
}


## 9. Contrôle de qualité des champs

Je vérifie sur les données réelles que l'extraction est fiable : combien de valeurs vides par
champ, et surtout que la clé unique n'a pas de doublons (sinon elle ne pourrait pas servir
d'identifiant).

In [8]:
total = len(resultats)
print(f"Total annonces : {total}\n")

print("Valeurs vides par champ :")
for champ in ["cle_unique", "reference", "intitule", "date_publication", "delai_soumission", "lien"]:
    vides = sum(1 for x in resultats if not x.get(champ))
    print(f"  {champ:18} : {vides} vide(s)")

cles = [x["cle_unique"] for x in resultats]
print(f"\nDoublons sur la cle_unique : {len(cles) - len(set(cles))}")

Total annonces : 22

Valeurs vides par champ :
  cle_unique         : 0 vide(s)
  reference          : 12 vide(s)
  intitule           : 0 vide(s)
  date_publication   : 0 vide(s)
  delai_soumission   : 0 vide(s)
  lien               : 0 vide(s)

Doublons sur la cle_unique : 0


## 10. Normalisation des dates

Les dates sont en français littéral (`16 juin 2026`). Sous cette forme on ne peut ni les comparer
ni calculer de délais. Je les convertis au format ISO `AAAA-MM-JJ`.

In [9]:
MOIS = {
    "janvier": 1, "février": 2, "fevrier": 2, "mars": 3, "avril": 4, "mai": 5,
    "juin": 6, "juillet": 7, "août": 8, "aout": 8, "septembre": 9,
    "octobre": 10, "novembre": 11, "décembre": 12, "decembre": 12,
}

def normaliser_date(txt):
    """'16 juin 2026' -> '2026-06-16'. Renvoie None si non reconnu."""
    if not txt:
        return None
    m = re.match(r"(\d{1,2})\s+(\w+)\s+(\d{4})", txt.strip().lower())
    if not m:
        return None
    jour, mois, annee = m.groups()
    num_mois = MOIS.get(mois)
    if not num_mois:
        return None
    return f"{annee}-{num_mois:02d}-{int(jour):02d}"

for x in resultats:
    x["date_publication"] = normaliser_date(x["date_publication"])
    x["delai_soumission"] = normaliser_date(x["delai_soumission"])

print("Exemple après normalisation :")
print(json.dumps(resultats[0], ensure_ascii=False, indent=2))

Exemple après normalisation :
{
  "cle_unique": "etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de",
  "reference": "",
  "intitule": "Etudes d’ingénierie et construction de l’ensemble immobilier de l’Agence Auxiliaire de Kandi au Bénin",
  "date_publication": "2026-06-16",
  "delai_soumission": "2026-07-30",
  "lien": "https://www.bceao.int/fr/appels-offres/etudes-dingenierie-et-construction-de-lensemble-immobilier-de-lagence-auxiliaire-de",
  "source": "BCEAO",
  "statut_source": "en cours"
}


## 11. Vérification : y a-t-il des annonces pertinentes ?

La BCEAO publie surtout des marchés de travaux et de fournitures. Je fais un filtre **très
grossier** par mots-clés, juste pour vérifier qu'il y a bien quelques marchés informatiques.

Attention : ce filtre par mots-clés est seulement une vérification rapide. Le vrai tri de
pertinence sera fait par le module de scoring (module 2), qui comprend le sens et saura distinguer
« audit de sécurité » (pertinent) de « refonte d'intranet » (développement web, hors périmètre).

In [10]:
mots_cles = ("informati", "numérique", "logiciel", "serveur", "site internet",
             "intranet", "cyber", "réseau", "fibre", "système d'information", "sécurité")

pertinentes = [x for x in resultats if any(m in x["intitule"].lower() for m in mots_cles)]

print(f"{len(pertinentes)} annonce(s) à connotation informatique :\n")
for x in pertinentes:
    print(" -", x["intitule"][:90])

2 annonce(s) à connotation informatique :

 - Report - Sélection d'un prestataire chargé de réaliser la refonte de l'intranet de la BCEA
 - Reprise du câblage fibre optique des immeubles fonctionnels du Siège


## 12. Sauvegarde

J'enregistre les annonces extraites en JSON. Ce sera la matière première à insérer dans la base
de données (MongoDB) à l'étape suivante.

In [11]:
with open("bceao_annonces.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, ensure_ascii=False, indent=2)

print("Sauvegardé :", len(resultats), "annonces dans bceao_annonces.json")

Sauvegardé : 22 annonces dans bceao_annonces.json


## 13. Conclusion et prochaines étapes

**Ce que cette exploration m'a appris :**

- La BCEAO est une source publique, accessible avec un simple `requests` (pas de navigateur
  headless), au moins pour la section « En cours ».
- La page est en sections (`<h2>`) ; je cible « En cours » via son titre, point d'ancrage fiable.
- Tout le contenu d'une annonce est dans le texte d'un `<a>`, d'où l'extraction par expression
  régulière, et le filtre « Publié le » pour écarter le décor.
- Deux nettoyages : clé = slug de l'URL (la référence manque parfois), dates converties en ISO.
- La source contient peu de cybersécurité : c'est un bon banc d'essai technique, et ça confirme
  l'utilité du scoring (module 2).

**Champs collectés :** `cle_unique`, `reference`, `intitule`, `date_publication`,
`delai_soumission`, `lien` (page de détail), `source`, `statut_source`.

**Prochaines étapes :**

1. Insérer ces annonces dans MongoDB avec déduplication sur `cle_unique`.
2. Scorer la pertinence (module 2) ; ne récupérer le détail (description, documents PDF) que pour
   les annonces jugées pertinentes (stratégie simple).
3. Pagination : parcourir les pages (`?page=N`) pour ne rien manquer.
4. Gérer la clôture des annonces : par la date limite (si dépassée = close) et par leur
   disparition de la section « En cours » lors des collectes suivantes.
5. Automatiser la collecte quotidienne avec n8n.